# Enterprise ETL Pipeline

## Project Information

| Item | Description |
|------|-------------|
| **Project** | E-Commerce Growth Analytics |
| **Dataset** | Olist Brazilian E-Commerce Public Dataset |
| **Notebook** | `04_etl_pipeline.ipynb` |
| **Author** | Paul Adeyemi |
| **Project Repository** | `ecommerce-growth-analytics` |

---

## Objective

Develop an enterprise-grade Extract, Transform, and Load (ETL) pipeline that prepares raw operational data for analytical processing.

This notebook transforms validated source datasets into clean, standardized, and analytics-ready datasets that will support feature engineering, SQL analytics, dashboard development, and business reporting.

---

## Input

The ETL pipeline consumes validated datasets extracted from the raw data layer.

Input datasets include:

- Customers
- Orders
- Order Items
- Payments
- Reviews
- Products
- Sellers
- Geolocation
- Product Category Translation

---

## Output

The notebook produces:

- Standardized datasets
- Cleaned analytical datasets
- Business-rule validated datasets
- Staging CSV files
- Transformation summaries

These outputs serve as the foundation for the Feature Engineering phase.

---

## Major ETL Activities

This notebook performs the following operations:

- Column standardization
- Data type conversion
- Missing value handling
- Duplicate handling
- Business rule validation
- Export to the staging layer

---

## Dependencies

Primary libraries used:

- pandas
- numpy
- pathlib
- os

---

## Expected Runtime

Approximately **2–5 minutes**, depending on the execution environment.

---

## Notes

This notebook follows enterprise data engineering practices, including:

- reusable transformation functions,
- business-driven data cleaning,
- comprehensive validation,
- modular code design,
- and detailed technical documentation.

The transformed datasets generated here become the primary input for the Feature Engineering stage.

# Phase 4: ETL Pipeline

## Objective

The purpose of this phase is to build the Extract, Transform, and Load (ETL) pipeline that converts the raw Olist operational datasets into a clean, standardized, and analytics-ready data warehouse.

The ETL pipeline follows the architecture defined during the Enterprise Data Warehouse Design phase.

The pipeline consists of three stages:

- Extract
- Transform
- Load

Every transformation will be reproducible, documented, and suitable for production environments.

# ETL Architecture

The pipeline is organized into three layers.

## Raw Layer

Contains the original datasets exactly as received.

No modifications are performed in this layer.

## Staging Layer

Responsible for:

- data cleaning
- missing value handling
- duplicate removal
- data type conversion
- feature standardization
- quality validation

## Warehouse Layer

Loads cleaned datasets into dimensional tables using surrogate keys and business rules.

# ETL Workflow

The pipeline will execute in the following sequence:

1. Extract raw datasets
2. Perform data validation
3. Clean and standardize records
4. Generate surrogate keys
5. Build dimension tables
6. Build fact tables
7. Export warehouse-ready datasets

In [35]:
from pathlib import Path
import pandas as pd

# Project directories
PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"
STAGING_PATH = PROJECT_ROOT / "data" / "staging"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

print("Raw:", RAW_PATH)
print("Staging:", STAGING_PATH)
print("Processed:", PROCESSED_PATH)

Raw: /Users/laplace/Documents/work/ecommerce-growth-analytics/data/raw
Staging: /Users/laplace/Documents/work/ecommerce-growth-analytics/data/staging
Processed: /Users/laplace/Documents/work/ecommerce-growth-analytics/data/processed


In [36]:
# Ensure output directories exist

STAGING_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Folders verified.")

Folders verified.


## 1. Confirm the source location

In [37]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

SOURCE_PATH = PROJECT_ROOT / "data" / "source" / "olist"
STAGING_PATH = PROJECT_ROOT / "data" / "staging"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

print("Source path:", SOURCE_PATH.resolve())
print("Source exists:", SOURCE_PATH.exists())

Source path: /Users/laplace/Documents/work/ecommerce-growth-analytics/data/source/olist
Source exists: True


## 2. Create the output folders

In [38]:
STAGING_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Staging and processed folders are ready.")

Staging and processed folders are ready.


## 3. Locate all source files

In [39]:
source_files = sorted(SOURCE_PATH.glob("*.csv"))

print(f"CSV files found: {len(source_files)}")

for file in source_files:
    print(file.name)

CSV files found: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


## 4. Build a reusable extraction function

In [40]:
def extract_csv_files(source_path: Path) -> dict[str, pd.DataFrame]:
    """
    Read every CSV file in the source directory.

    Returns:
        Dictionary where each key is the filename without extension
        and each value is the loaded pandas DataFrame.
    """
    extracted_data = {}

    csv_files = sorted(source_path.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files were found in: {source_path.resolve()}"
        )

    for file in csv_files:
        extracted_data[file.stem] = pd.read_csv(file)

    return extracted_data

## 5. Extract the datasets

In [41]:
raw_datasets = extract_csv_files(SOURCE_PATH)

print(f"Successfully extracted {len(raw_datasets)} datasets.")

Successfully extracted 9 datasets.


## 6. Validate the extraction

In [42]:
extraction_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": df.shape[0],
            "columns": df.shape[1]
        }
        for name, df in raw_datasets.items()
    ]
).sort_values("rows", ascending=False)

extraction_summary

,dataset,rows,columns
1,olist_geolocation_dataset,1000163,5
2,olist_order_items_dataset,112650,7
3,olist_order_payments_dataset,103886,5
0,olist_customers_dataset,99441,5
5,olist_orders_dataset,99441,8
4,olist_order_reviews_dataset,99224,7
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


## Extract Stage Result

All nine Olist source datasets were loaded successfully from the immutable source layer.

The extracted row and column counts were compared with the Phase 1 source audit to confirm that no records were lost during ingestion.

No transformations have been applied at this stage.

# Data Validation Stage

Before any transformation is applied, each dataset must be validated.

The validation stage ensures that the source data satisfies the minimum quality requirements needed for analytical processing.

Validation is performed before cleaning to preserve the integrity of the original datasets and to document any issues detected during ingestion.

The validation process includes:

- Dataset dimensions
- Missing values
- Duplicate records
- Data types
- Memory usage
- Primary key validation
- Foreign key validation

In [43]:
def dataset_overview(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    """
    Return a high-level overview of a dataset.
    """

    summary = pd.DataFrame({
        "Dataset": [dataset_name],
        "Rows": [df.shape[0]],
        "Columns": [df.shape[1]],
        "Memory (MB)": [round(df.memory_usage(deep=True).sum()/1024**2,2)]
    })

    return summary

In [44]:
dataset_overview(
    raw_datasets["olist_orders_dataset"],
    "Orders"
)

,Dataset,Rows,Columns,Memory (MB)
0,Orders,99441,8,52.94


## Missing Value Function

In [45]:
def missing_value_summary(df: pd.DataFrame) -> pd.DataFrame:

    missing = df.isna().sum()

    result = (
        pd.DataFrame({
            "Missing Values": missing,
            "Percentage": round(
                (missing / len(df)) * 100,
                2
            )
        })
        .query("`Missing Values` > 0")
        .sort_values(
            "Missing Values",
            ascending=False
        )
    )

    return result

In [46]:
missing_value_summary(
    raw_datasets["olist_orders_dataset"]
)

,Missing Values,Percentage
order_delivered_customer_date,2965,2.98
order_delivered_carrier_date,1783,1.79
order_approved_at,160,0.16


### Duplicate Function

In [47]:
def duplicate_summary(df: pd.DataFrame):

    duplicates = df.duplicated().sum()

    return pd.DataFrame({

        "Duplicate Rows":[duplicates],

        "Percentage":[
            round(
                duplicates/len(df)*100,
                2
            )
        ]

    })

In [48]:
duplicate_summary(
    raw_datasets["olist_products_dataset"]
)

,Duplicate Rows,Percentage
0,0,0.0


# Primary Key Validation

Primary keys uniquely identify every record in a table.

A valid primary key must satisfy two conditions:

- It cannot contain missing values.
- It cannot contain duplicate values.

Primary key validation is performed before any transformation to ensure that the source data maintains entity integrity.

In [49]:
def validate_primary_key(df: pd.DataFrame, primary_key: str) -> pd.DataFrame:
    """
    Validate a primary key column.

    Checks:
    - Missing values
    - Duplicate values
    """

    missing = df[primary_key].isna().sum()
    duplicates = df[primary_key].duplicated().sum()

    return pd.DataFrame({
        "Primary Key": [primary_key],
        "Missing Values": [missing],
        "Duplicate Values": [duplicates],
        "Status": [
            "PASS"
            if missing == 0 and duplicates == 0
            else "FAIL"
        ]
    })

In [50]:
validate_primary_key(
    raw_datasets["olist_customers_dataset"],
    "customer_id"
)

,Primary Key,Missing Values,Duplicate Values,Status
0,customer_id,0,0,PASS


# Foreign Key Validation

Foreign keys establish relationships between tables.

Every foreign key should reference an existing record in the related parent table.

Records with missing parent records are known as orphan records and should be investigated before loading the warehouse.

In [51]:
def validate_foreign_key(
    child_df: pd.DataFrame,
    parent_df: pd.DataFrame,
    foreign_key: str,
    parent_key: str
) -> pd.DataFrame:
    """
    Validate foreign key integrity.
    """

    invalid = ~child_df[foreign_key].isin(parent_df[parent_key])

    orphan_count = invalid.sum()

    return pd.DataFrame({
        "Foreign Key":[foreign_key],
        "Parent Table Key":[parent_key],
        "Orphan Records":[orphan_count],
        "Status":[
            "PASS"
            if orphan_count == 0
            else "FAIL"
        ]
    })

In [52]:
validate_foreign_key(
    raw_datasets["olist_orders_dataset"],
    raw_datasets["olist_customers_dataset"],
    "customer_id",
    "customer_id"
)

,Foreign Key,Parent Table Key,Orphan Records,Status
0,customer_id,customer_id,0,PASS


# Data Type Validation

Correct data types are essential for analytical processing.

This validation compares the detected data types with the expected business data types before transformation.

Incorrect data types may lead to calculation errors, failed joins, and inconsistent reporting.

In [53]:
def data_type_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Display column data types.
    """

    return pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values
    })

In [54]:
data_type_summary(
    raw_datasets["olist_orders_dataset"]
)

,Column,Data Type
0,order_id,str
1,customer_id,str
2,order_status,str
3,order_purchase_timestamp,str
4,order_approved_at,str
5,order_delivered_carrier_date,str
6,order_delivered_customer_date,str
7,order_estimated_delivery_date,str


# Consolidated Validation Report

The individual validation checks will now be combined into a single summary report.

This report provides a high-level view of dataset quality before transformation begins.

The report will include:

- row count,
- column count,
- memory usage,
- missing-value count,
- duplicate-row count,
- primary-key status,
- and overall validation status.

The consolidated report acts as a quality gate between the Extract and Transform stages.

### Report building

In [55]:
def build_validation_report(
    datasets: dict[str, pd.DataFrame],
    primary_keys: dict[str, str]
) -> pd.DataFrame:
    """
    Build a consolidated validation report for multiple datasets.

    Parameters
    ----------
    datasets:
        Dictionary of dataset names and DataFrames.

    primary_keys:
        Dictionary mapping dataset names to their primary key columns.

    Returns
    -------
    pd.DataFrame
        Consolidated validation report.
    """

    report_rows = []

    for dataset_name, df in datasets.items():

        row_count = df.shape[0]
        column_count = df.shape[1]
        memory_mb = round(
            df.memory_usage(deep=True).sum() / 1024**2,
            2
        )

        missing_cells = int(df.isna().sum().sum())
        duplicate_rows = int(df.duplicated().sum())

        primary_key = primary_keys.get(dataset_name)

        if primary_key and primary_key in df.columns:
            pk_missing = int(df[primary_key].isna().sum())
            pk_duplicates = int(df[primary_key].duplicated().sum())

            primary_key_status = (
                "PASS"
                if pk_missing == 0 and pk_duplicates == 0
                else "FAIL"
            )
        else:
            pk_missing = None
            pk_duplicates = None
            primary_key_status = "NOT DEFINED"

        overall_status = (
            "PASS"
            if duplicate_rows == 0
            and primary_key_status in {"PASS", "NOT DEFINED"}
            else "REVIEW"
        )

        report_rows.append({
            "dataset": dataset_name,
            "rows": row_count,
            "columns": column_count,
            "memory_mb": memory_mb,
            "missing_cells": missing_cells,
            "duplicate_rows": duplicate_rows,
            "primary_key": primary_key,
            "pk_missing": pk_missing,
            "pk_duplicates": pk_duplicates,
            "primary_key_status": primary_key_status,
            "overall_status": overall_status
        })

    return (
        pd.DataFrame(report_rows)
        .sort_values("rows", ascending=False)
        .reset_index(drop=True)
    )

### Define the primary keys

In [56]:
primary_keys = {
    "olist_customers_dataset": "customer_id",
    "olist_orders_dataset": "order_id",
    "olist_products_dataset": "product_id",
    "olist_sellers_dataset": "seller_id",
    "olist_order_reviews_dataset": "review_id",
    "product_category_name_translation": "product_category_name"
}

### Generate the report

In [57]:
validation_report = build_validation_report(
    raw_datasets,
    primary_keys
)

validation_report

,dataset,rows,columns,memory_mb,missing_cells,duplicate_rows,primary_key,pk_missing,pk_duplicates,primary_key_status,overall_status
0,olist_geolocation_dataset,1000163,5,129.38,0,261831,NaN,NaN,NaN,NOT DEFINED,REVIEW
1,olist_order_items_dataset,112650,7,35.99,0,0,NaN,NaN,NaN,NOT DEFINED,PASS
2,olist_order_payments_dataset,103886,5,16.23,0,0,NaN,NaN,NaN,NOT DEFINED,PASS
3,olist_customers_dataset,99441,5,26.59,0,0,customer_id,0.0,0.0,PASS,PASS
4,olist_orders_dataset,99441,8,52.94,4908,0,order_id,0.0,0.0,PASS,PASS
5,olist_order_reviews_dataset,99224,7,39.12,145903,0,review_id,0.0,814.0,FAIL,REVIEW
6,olist_products_dataset,32951,9,6.30,2448,0,product_id,0.0,0.0,PASS,PASS
7,olist_sellers_dataset,3095,4,0.59,0,0,seller_id,0.0,0.0,PASS,PASS
8,product_category_name_translation,71,2,0.01,0,0,product_category_name,0.0,0.0,PASS,PASS


### Save the report

In [58]:
validation_report_path = (
    STAGING_PATH / "source_validation_report.csv"
)

validation_report.to_csv(
    validation_report_path,
    index=False
)

print(
    f"Validation report saved to: "
    f"{validation_report_path.resolve()}"
)

Validation report saved to: /Users/laplace/Documents/work/ecommerce-growth-analytics/data/staging/source_validation_report.csv


## Validation Stage Result

The source datasets were evaluated for structural integrity, missing values, duplicate rows, memory usage, and primary-key quality.

The consolidated validation report has been exported to the staging layer for traceability.

Datasets marked as `PASS` satisfy the initial validation requirements.

Datasets marked as `REVIEW` require further investigation during the transformation stage.

No source data has been modified during validation.

In [59]:
validation_report

,dataset,rows,columns,memory_mb,missing_cells,duplicate_rows,primary_key,pk_missing,pk_duplicates,primary_key_status,overall_status
0,olist_geolocation_dataset,1000163,5,129.38,0,261831,NaN,NaN,NaN,NOT DEFINED,REVIEW
1,olist_order_items_dataset,112650,7,35.99,0,0,NaN,NaN,NaN,NOT DEFINED,PASS
2,olist_order_payments_dataset,103886,5,16.23,0,0,NaN,NaN,NaN,NOT DEFINED,PASS
3,olist_customers_dataset,99441,5,26.59,0,0,customer_id,0.0,0.0,PASS,PASS
4,olist_orders_dataset,99441,8,52.94,4908,0,order_id,0.0,0.0,PASS,PASS
5,olist_order_reviews_dataset,99224,7,39.12,145903,0,review_id,0.0,814.0,FAIL,REVIEW
6,olist_products_dataset,32951,9,6.30,2448,0,product_id,0.0,0.0,PASS,PASS
7,olist_sellers_dataset,3095,4,0.59,0,0,seller_id,0.0,0.0,PASS,PASS
8,product_category_name_translation,71,2,0.01,0,0,product_category_name,0.0,0.0,PASS,PASS


In [60]:
validation_report.sort_values(
    by="overall_status",
    ascending=False
)

,dataset,rows,columns,memory_mb,missing_cells,duplicate_rows,primary_key,pk_missing,pk_duplicates,primary_key_status,overall_status
0,olist_geolocation_dataset,1000163,5,129.38,0,261831,NaN,NaN,NaN,NOT DEFINED,REVIEW
5,olist_order_reviews_dataset,99224,7,39.12,145903,0,review_id,0.0,814.0,FAIL,REVIEW
1,olist_order_items_dataset,112650,7,35.99,0,0,NaN,NaN,NaN,NOT DEFINED,PASS
2,olist_order_payments_dataset,103886,5,16.23,0,0,NaN,NaN,NaN,NOT DEFINED,PASS
3,olist_customers_dataset,99441,5,26.59,0,0,customer_id,0.0,0.0,PASS,PASS
4,olist_orders_dataset,99441,8,52.94,4908,0,order_id,0.0,0.0,PASS,PASS
6,olist_products_dataset,32951,9,6.30,2448,0,product_id,0.0,0.0,PASS,PASS
7,olist_sellers_dataset,3095,4,0.59,0,0,seller_id,0.0,0.0,PASS,PASS
8,product_category_name_translation,71,2,0.01,0,0,product_category_name,0.0,0.0,PASS,PASS


In [61]:
duplicate_reviews = (
    raw_datasets["olist_order_reviews_dataset"]
    .loc[
        raw_datasets["olist_order_reviews_dataset"]["review_id"].duplicated(keep=False)
    ]
    .sort_values("review_id")
)

duplicate_reviews.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [62]:
duplicate_reviews["review_id"].nunique()

789

In [63]:
len(duplicate_reviews)

1603

# Transformation Stage

The transformation stage converts validated operational datasets into standardized, analytics-ready datasets.

Unlike the validation stage, transformations modify the data according to predefined business rules.

The objectives of this stage are:

- Standardize column names
- Convert data types
- Handle missing values
- Remove invalid duplicates
- Apply business rules
- Engineer analytical features
- Export cleaned datasets to the staging layer

## Column Standardization

Column names should be consistent across all datasets.

The transformation process will:

- convert names to lowercase,
- replace spaces with underscores,
- remove unnecessary characters,
- preserve business meaning,
- improve readability for downstream analytics.

### First Transformation Function

In [64]:
def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize DataFrame column names.

    Rules:
    - lowercase
    - strip whitespace
    - replace spaces with underscores
    - replace hyphens with underscores
    """

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    return df

In [65]:
orders = standardize_column_names(
    raw_datasets["olist_orders_dataset"]
)

orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

### Schema Validation After Standardization

In [66]:
def compare_column_names(
    original_df: pd.DataFrame,
    transformed_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Compare original and standardized column names.
    """

    return pd.DataFrame({
        "Original": original_df.columns,
        "Standardized": transformed_df.columns
    })

In [67]:
compare_column_names(
    raw_datasets["olist_orders_dataset"],
    orders
)

,Original,Standardized
0,order_id,order_id
1,customer_id,customer_id
2,order_status,order_status
3,order_purchase_timestamp,order_purchase_timestamp
4,order_approved_at,order_approved_at
5,order_delivered_carrier_date,order_delivered_carrier_date
6,order_delivered_customer_date,order_delivered_customer_date
7,order_estimated_delivery_date,order_estimated_delivery_date


# Transformation Stage

The transformation stage converts validated source datasets into standardized, analytics-ready datasets.

During this stage, the pipeline will:

- standardize column names,
- correct known source naming errors,
- convert data types,
- handle missing values according to business rules,
- resolve invalid duplicates,
- and prepare cleaned datasets for the staging layer.

In [68]:
# ============================================
# Column Standardization Function
# ============================================

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy of a DataFrame with standardized column names.

    Rules:
    - remove leading and trailing whitespace,
    - convert names to lowercase,
    - replace spaces with underscores,
    - replace hyphens with underscores.
    """
    transformed_df = df.copy()

    transformed_df.columns = (
        transformed_df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    return transformed_df

In [69]:
# ============================================
# Apply Column Standardization
# ============================================

transformed_datasets = {
    dataset_name: standardize_column_names(df)
    for dataset_name, df in raw_datasets.items()
}

print(
    f"Column names standardized for "
    f"{len(transformed_datasets)} datasets."
)

Column names standardized for 9 datasets.


In [70]:
# ============================================
# Compare Original and Standardized Columns
# ============================================

def compare_column_names(
    original_df: pd.DataFrame,
    transformed_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Compare source column names with standardized column names.
    """
    return pd.DataFrame({
        "original_column": original_df.columns,
        "standardized_column": transformed_df.columns
    })

In [71]:
# ============================================
# Verify Orders Column Standardization
# ============================================

compare_column_names(
    raw_datasets["olist_orders_dataset"],
    transformed_datasets["olist_orders_dataset"]
)

,original_column,standardized_column
0,order_id,order_id
1,customer_id,customer_id
2,order_status,order_status
3,order_purchase_timestamp,order_purchase_timestamp
4,order_approved_at,order_approved_at
5,order_delivered_carrier_date,order_delivered_carrier_date
6,order_delivered_customer_date,order_delivered_customer_date
7,order_estimated_delivery_date,order_estimated_delivery_date


In [72]:
# ============================================
# Correct Product Column Spelling Errors
# ============================================

products = transformed_datasets[
    "olist_products_dataset"
].copy()

products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": (
        "product_description_length"
    )
})

transformed_datasets[
    "olist_products_dataset"
] = products

In [73]:
# ============================================
# Verify Corrected Product Column Names
# ============================================

transformed_datasets[
    "olist_products_dataset"
].columns.tolist()

['product_id',
 'product_category_name',
 'product_name_length',
 'product_description_length',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

## Column Standardization Result

Column names were standardized across all nine datasets using a reusable transformation function.

The Olist source datasets already followed a mostly consistent lowercase naming convention. However, two known spelling errors in the product dataset were corrected:

- `product_name_lenght` → `product_name_length`
- `product_description_lenght` → `product_description_length`

No business records or values were modified during this transformation.

# Data Type Conversion

The source datasets contain several columns stored as text that represent dates and timestamps.

These columns will be converted into native Pandas datetime objects to support:

- time-based calculations,
- feature engineering,
- trend analysis,
- and analytical reporting.

Invalid datetime values will be converted to missing values (`NaT`) using safe parsing.

In [74]:
# ============================================
# Datetime Conversion Function
# ============================================

def convert_datetime_columns(
    df: pd.DataFrame,
    columns: list[str]
) -> pd.DataFrame:
    """
    Convert specified columns to datetime.

    Parameters
    ----------
    df:
        Source DataFrame.

    columns:
        List of datetime columns.

    Returns
    -------
    pd.DataFrame
        DataFrame with converted datetime columns.
    """

    transformed_df = df.copy()

    for column in columns:

        transformed_df[column] = pd.to_datetime(
            transformed_df[column],
            errors="coerce"
        )

    return transformed_df

In [75]:
# ============================================
# Datetime Columns
# ============================================

order_datetime_columns = [

    "order_purchase_timestamp",

    "order_approved_at",

    "order_delivered_carrier_date",

    "order_delivered_customer_date",

    "order_estimated_delivery_date"

]

review_datetime_columns = [

    "review_creation_date",

    "review_answer_timestamp"

]

In [76]:
# ============================================
# Convert Orders Datetime Columns
# ============================================

transformed_datasets["olist_orders_dataset"] = convert_datetime_columns(

    transformed_datasets["olist_orders_dataset"],

    order_datetime_columns

)

In [77]:
# ============================================
# Convert Review Datetime Columns
# ============================================

transformed_datasets["olist_order_reviews_dataset"] = convert_datetime_columns(

    transformed_datasets["olist_order_reviews_dataset"],

    review_datetime_columns

)

In [78]:
# ============================================
# Verify Datetime Conversion
# ============================================

transformed_datasets["olist_orders_dataset"].dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

# Missing Value Handling

Missing values require business-driven decisions rather than blanket removal or imputation.

Each missing value is evaluated based on its business meaning before an appropriate strategy is applied.

The objectives of this step are:

- identify acceptable missing values,
- define column-specific handling strategies,
- preserve business information,
- and prepare datasets for downstream analytics.

In [79]:
# ============================================
# Missing Value Handling Strategy
# ============================================

In [84]:
# ============================================
# Missing Value Handling Strategy
# ============================================

missing_value_strategies = {
    "olist_order_reviews_dataset": {
        "review_comment_title": {
            "action": "fill",
            "value": "No title"
        },
        "review_comment_message": {
            "action": "fill",
            "value": "No comment"
        }
    },

    "olist_products_dataset": {
        "product_category_name": {
            "action": "fill",
            "value": "unknown"
        },
        "product_name_length": {
            "action": "preserve"
        },
        "product_description_length": {
            "action": "preserve"
        },
        "product_photos_qty": {
            "action": "preserve"
        },
        "product_weight_g": {
            "action": "preserve"
        },
        "product_length_cm": {
            "action": "preserve"
        },
        "product_height_cm": {
            "action": "preserve"
        },
        "product_width_cm": {
            "action": "preserve"
        }
    },

    "olist_orders_dataset": {
        "order_approved_at": {
            "action": "preserve"
        },
        "order_delivered_carrier_date": {
            "action": "preserve"
        },
        "order_delivered_customer_date": {
            "action": "preserve"
        }
    }
}

In [85]:
# ============================================
# Missing Value Handling Function
# ============================================

def handle_missing_values(
    df: pd.DataFrame,
    strategy: dict[str, dict]
) -> pd.DataFrame:
    """
    Apply column-specific missing-value rules to a DataFrame.

    Supported actions:
    - preserve: leave missing values unchanged
    - fill: replace missing values with a configured value
    - median: replace missing numeric values with the column median
    - mode: replace missing values with the most frequent value
    - drop_rows: remove rows where the column is missing

    Parameters
    ----------
    df:
        Source DataFrame.

    strategy:
        Dictionary containing the action and optional replacement
        value for each column.

    Returns
    -------
    pd.DataFrame
        A transformed copy of the DataFrame.
    """
    transformed_df = df.copy()

    for column, rule in strategy.items():

        if column not in transformed_df.columns:
            raise KeyError(
                f"Column '{column}' does not exist in the DataFrame."
            )

        action = rule.get("action")

        if action == "preserve":
            continue

        if action == "fill":
            if "value" not in rule:
                raise ValueError(
                    f"No fill value was supplied for '{column}'."
                )

            transformed_df[column] = (
                transformed_df[column].fillna(rule["value"])
            )

        elif action == "median":
            if not pd.api.types.is_numeric_dtype(
                transformed_df[column]
            ):
                raise TypeError(
                    f"Column '{column}' must be numeric "
                    "to use the median strategy."
                )

            transformed_df[column] = transformed_df[column].fillna(
                transformed_df[column].median()
            )

        elif action == "mode":
            mode_values = transformed_df[column].mode(dropna=True)

            if mode_values.empty:
                raise ValueError(
                    f"No mode could be calculated for '{column}'."
                )

            transformed_df[column] = transformed_df[column].fillna(
                mode_values.iloc[0]
            )

        elif action == "drop_rows":
            transformed_df = transformed_df.dropna(
                subset=[column]
            )

        else:
            raise ValueError(
                f"Unsupported action '{action}' for column '{column}'."
            )

    return transformed_df

In [86]:
# ============================================
# Apply Missing Value Strategies
# ============================================

for dataset_name, strategy in missing_value_strategies.items():

    transformed_datasets[dataset_name] = handle_missing_values(
        transformed_datasets[dataset_name],
        strategy
    )

print(
    "Missing-value strategies applied to "
    f"{len(missing_value_strategies)} datasets."
)

Missing-value strategies applied to 3 datasets.


In [87]:
# ============================================
# Verify Missing Value Handling
# ============================================

missing_value_verification = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "missing_cells_after": int(
                df.isna().sum().sum()
            ),
            "rows_after": len(df)
        }
        for dataset_name, df in transformed_datasets.items()
    ]
).sort_values(
    "missing_cells_after",
    ascending=False
).reset_index(drop=True)

missing_value_verification

,dataset,missing_cells_after,rows_after
0,olist_orders_dataset,4908,99441
1,olist_products_dataset,1838,32951
2,olist_customers_dataset,0,99441
3,olist_geolocation_dataset,0,1000163
4,olist_order_items_dataset,0,112650
5,olist_order_payments_dataset,0,103886
6,olist_order_reviews_dataset,0,99224
7,olist_sellers_dataset,0,3095
8,product_category_name_translation,0,71


In [88]:
# ============================================
# Verify Review Text Replacements
# ============================================

transformed_datasets[
    "olist_order_reviews_dataset"
][
    [
        "review_comment_title",
        "review_comment_message"
    ]
].isna().sum()

review_comment_title      0
review_comment_message    0
dtype: int64

## Missing Value Handling Result

Missing values were handled using dataset-specific business rules rather than a blanket deletion or imputation approach.

The following decisions were applied:

- Missing review titles were replaced with `No title`.
- Missing review messages were replaced with `No comment`.
- Missing operational order timestamps were preserved because they may reflect cancelled, unavailable, or incomplete process stages.
- Missing product categories were labelled `unknown`.
- Other missing product attributes were preserved because reliable replacement values could not be inferred from the source data.

No rows were removed during this step.

# Duplicate Handling

Duplicate records must be evaluated according to the grain and business meaning of each dataset.

Repeated business identifiers are not automatically considered duplicates because they may represent valid one-to-many relationships.

This step will:

- identify exact duplicate rows,
- validate composite business keys,
- preserve legitimate repeated identifiers,
- and remove only records that are confirmed to be invalid duplicates.

In [89]:
# ============================================
# Composite Key Validation Function
# ============================================

def validate_composite_key(
    df: pd.DataFrame,
    key_columns: list[str]
) -> pd.DataFrame:
    """
    Validate whether a combination of columns uniquely identifies rows.

    Parameters
    ----------
    df:
        Source DataFrame.

    key_columns:
        Columns that together form the expected composite key.

    Returns
    -------
    pd.DataFrame
        Summary of missing and duplicated composite-key records.
    """
    missing_key_rows = int(
        df[key_columns].isna().any(axis=1).sum()
    )

    duplicate_key_rows = int(
        df.duplicated(
            subset=key_columns,
            keep=False
        ).sum()
    )

    status = (
        "PASS"
        if missing_key_rows == 0 and duplicate_key_rows == 0
        else "REVIEW"
    )

    return pd.DataFrame({
        "key_columns": [" + ".join(key_columns)],
        "missing_key_rows": [missing_key_rows],
        "duplicate_key_rows": [duplicate_key_rows],
        "status": [status]
    })

In [90]:
# ============================================
# Validate Order Items Composite Key
# ============================================

order_items_key_validation = validate_composite_key(
    transformed_datasets["olist_order_items_dataset"],
    ["order_id", "order_item_id"]
)

order_items_key_validation

,key_columns,missing_key_rows,duplicate_key_rows,status
0,order_id + order_item_id,0,0,PASS


In [91]:
# ============================================
# Validate Payments Composite Key
# ============================================

payments_key_validation = validate_composite_key(
    transformed_datasets["olist_order_payments_dataset"],
    ["order_id", "payment_sequential"]
)

payments_key_validation

,key_columns,missing_key_rows,duplicate_key_rows,status
0,order_id + payment_sequential,0,0,PASS


In [92]:
# ============================================
# Exact Duplicate Row Summary
# ============================================

duplicate_row_summary = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "exact_duplicate_rows": int(
                df.duplicated().sum()
            )
        }
        for dataset_name, df in transformed_datasets.items()
    ]
).sort_values(
    "exact_duplicate_rows",
    ascending=False
).reset_index(drop=True)

duplicate_row_summary

,dataset,exact_duplicate_rows
0,olist_geolocation_dataset,261831
1,olist_customers_dataset,0
2,olist_order_items_dataset,0
3,olist_order_payments_dataset,0
4,olist_order_reviews_dataset,0
5,olist_orders_dataset,0
6,olist_products_dataset,0
7,olist_sellers_dataset,0
8,product_category_name_translation,0


In [93]:
# ============================================
# Build Standardized Geolocation Dataset
# ============================================

geolocation = transformed_datasets[
    "olist_geolocation_dataset"
].copy()

geolocation_standardized = (
    geolocation
    .groupby(
        "geolocation_zip_code_prefix",
        as_index=False
    )
    .agg({
        "geolocation_lat": "median",
        "geolocation_lng": "median",
        "geolocation_city": (
            lambda values: values.mode().iloc[0]
            if not values.mode().empty
            else "unknown"
        ),
        "geolocation_state": (
            lambda values: values.mode().iloc[0]
            if not values.mode().empty
            else "unknown"
        )
    })
)

In [94]:
# ============================================
# Rename Standardized Geolocation Columns
# ============================================

geolocation_standardized = (
    geolocation_standardized.rename(columns={
        "geolocation_zip_code_prefix": "zip_code_prefix",
        "geolocation_lat": "latitude",
        "geolocation_lng": "longitude",
        "geolocation_city": "city",
        "geolocation_state": "state"
    })
)

transformed_datasets[
    "olist_geolocation_dataset"
] = geolocation_standardized

In [95]:
# ============================================
# Verify Geolocation Transformation
# ============================================

pd.DataFrame({
    "metric": [
        "source_rows",
        "standardized_rows",
        "unique_zip_codes",
        "duplicate_rows_after"
    ],
    "value": [
        len(raw_datasets["olist_geolocation_dataset"]),
        len(geolocation_standardized),
        geolocation_standardized["zip_code_prefix"].nunique(),
        int(geolocation_standardized.duplicated().sum())
    ]
})

,metric,value
0,source_rows,1000163
1,standardized_rows,19015
2,unique_zip_codes,19015
3,duplicate_rows_after,0


## Duplicate Handling Result

Duplicate handling was performed according to each dataset's grain.

The following findings and actions were recorded:

- `order_id + order_item_id` uniquely identifies each order-item record.
- `order_id + payment_sequential` uniquely identifies each payment transaction.
- Repeated `review_id` values were preserved because they represent valid review records associated with different orders.
- The raw geolocation dataset contained repeated observations for ZIP-code prefixes.
- Geolocation records were aggregated to one representative row per ZIP-code prefix using median coordinates and the most frequent city and state.
- No valid order, payment, review, customer, product, seller, or category records were removed.

# Business Rule Validation

After transformation, the datasets are validated against business rules to ensure that the cleaned data is suitable for analytical processing.

The objective of this step is to verify that the transformation stage has produced datasets that satisfy the agreed business requirements before they are exported to the staging layer.

In [96]:
# ============================================
# Business Rule Validation Function
# ============================================

def validate_business_rules(
    transformed_datasets: dict[str, pd.DataFrame]
) -> pd.DataFrame:
    """
    Validate selected business rules after transformation.

    Returns
    -------
    pd.DataFrame
        Summary of business-rule validation.
    """

    validation_results = []

    # Orders
    orders = transformed_datasets["olist_orders_dataset"]

    validation_results.append({

        "dataset": "olist_orders_dataset",

        "rule": "Purchase timestamp exists",

        "status":
            "PASS"
            if orders["order_purchase_timestamp"].isna().sum() == 0
            else "FAIL"

    })

    # Products
    products = transformed_datasets["olist_products_dataset"]

    validation_results.append({

        "dataset": "olist_products_dataset",

        "rule": "Product category available or labelled unknown",

        "status":
            "PASS"
            if products["product_category_name"].isna().sum() == 0
            else "FAIL"

    })

    # Reviews
    reviews = transformed_datasets["olist_order_reviews_dataset"]

    validation_results.append({

        "dataset": "olist_order_reviews_dataset",

        "rule": "Review comments contain no missing values",

        "status":
            "PASS"
            if (
                reviews["review_comment_title"].isna().sum() == 0
                and
                reviews["review_comment_message"].isna().sum() == 0
            )
            else "FAIL"

    })

    return pd.DataFrame(validation_results)

In [97]:
# ============================================
# Run Business Rule Validation
# ============================================

business_rule_validation = validate_business_rules(
    transformed_datasets
)

business_rule_validation

,dataset,rule,status
0,olist_orders_dataset,Purchase timestamp exists,PASS
1,olist_products_dataset,Product category available or labelled unknown,PASS
2,olist_order_reviews_dataset,Review comments contain no missing values,PASS


# Export Staging Datasets

The transformed datasets are exported to the staging layer.

These datasets will serve as the analytical source for feature engineering, SQL analytics, and dashboard development.

In [98]:
# ============================================
# Export Staging Datasets
# ============================================

for dataset_name, df in transformed_datasets.items():

    output_path = (
        STAGING_PATH /
        f"{dataset_name}.csv"
    )

    df.to_csv(
        output_path,
        index=False
    )

print(
    f"{len(transformed_datasets)} datasets exported "
    "to the staging layer."
)

9 datasets exported to the staging layer.


In [99]:
# ============================================
# Verify Staging Files
# ============================================

import os

sorted(
    os.listdir(STAGING_PATH)
)

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv',
 'source_validation_report.csv']

## Transformation Stage Result

The transformation stage successfully standardized, validated, and prepared the source datasets for analytical processing.

The completed transformations included:

- Column standardization
- Datetime conversion
- Missing value handling
- Duplicate handling
- Business rule validation
- Staging data export

The transformed datasets have been saved to the staging layer and will be used as the source for feature engineering in the next phase.

# ETL Pipeline Summary

## Overview

The Enterprise ETL Pipeline has successfully extracted, validated, transformed, and prepared the Olist datasets for analytical processing.

The resulting staging datasets are clean, standardized, and ready for feature engineering.

---

## Completed Activities

### Extraction

- Loaded all source datasets.
- Configured project paths.
- Created reusable dataset extraction functions.

---

### Validation

- Assessed dataset structure.
- Evaluated missing values.
- Checked duplicate records.
- Validated primary keys.
- Investigated composite keys.
- Produced a consolidated validation report.

---

### Transformation

- Standardized column names.
- Corrected source naming inconsistencies.
- Converted datetime columns.
- Applied business-driven missing value strategies.
- Investigated duplicate records.
- Standardized geolocation data.
- Validated business rules.

---

### Load

- Exported transformed datasets to the staging layer.
- Verified successful export of all staging files.

---

## Deliverables

The ETL pipeline produced:

- Analytics-ready staging datasets
- Validation reports
- Business rule validation reports
- Standardized geolocation dataset
- Reusable transformation functions

---

## Next Phase

The staging datasets generated in this notebook will serve as the input for:

# Phase 5 — Feature Engineering

During the next phase, analytical features will be engineered to support business intelligence, SQL analytics, dashboard development, and predictive modeling.

In [100]:
# ============================================
# ETL Pipeline Completion
# ============================================

print("=" * 60)
print("Enterprise ETL Pipeline Completed Successfully")
print("=" * 60)
print(f"Datasets Processed : {len(transformed_datasets)}")
print(f"Datasets Exported  : {len(transformed_datasets)}")
print("Next Phase         : Feature Engineering")
print("=" * 60)

Enterprise ETL Pipeline Completed Successfully
Datasets Processed : 9
Datasets Exported  : 9
Next Phase         : Feature Engineering
